# RAG

## 1. Database

### Отримання даних

В якості бази даних для LLM будемо використовувати FAQ курсів Data Talks Club. Вони оброблені і готові до використання та доступні на сайті в форматі JSON. Отримаємо їх.

In [1]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

Цей код поверне нам список курсів від Data Talks Club. Виведемо його і подивимось. 

In [2]:
import json
print(json.dumps(courses_raw, ensure_ascii=False, indent=2))

[
  {
    "course": "data-engineering-zoomcamp",
    "course_name": "Data Engineering Zoomcamp",
    "path": "/json/data-engineering-zoomcamp.json",
    "questions_count": 404
  },
  {
    "course": "stock-markets-analytics-zoomcamp",
    "course_name": "Stock Markets Analytics Zoomcamp",
    "path": "/json/stock-markets-analytics-zoomcamp.json",
    "questions_count": 93
  },
  {
    "course": "ai-dev-tools-zoomcamp",
    "course_name": "AI Dev Tools Zoomcamp",
    "path": "/json/ai-dev-tools-zoomcamp.json",
    "questions_count": 41
  },
  {
    "course": "llm-zoomcamp",
    "course_name": "LLM Zoomcamp",
    "path": "/json/llm-zoomcamp.json",
    "questions_count": 85
  },
  {
    "course": "mlops-zoomcamp",
    "course_name": "MLOps Zoomcamp",
    "path": "/json/mlops-zoomcamp.json",
    "questions_count": 255
  },
  {
    "course": "machine-learning-zoomcamp",
    "course_name": "ML Zoomcamp",
    "path": "/json/machine-learning-zoomcamp.json",
    "questions_count": 472
  }
]


Кожний запис має поле PATH, що містить шлях до FAQу конкретного курсу. Отримаємо тепер їх усі.

In [2]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1350

Ось що відбувається у коді в попередній комірці.

1. Функція requests.get() надсилає запит на вказану адресу (course_url), щоб отримати дані (зазвичай у форматі JSON). Результатом виконання є об'єкт відповіді (Response), який зберігається у змінну course_response. Цей об'єкт містить статус відповіді, заголовки, тіло відповіді тощо.

2. Метод .raise_for_status() перевіряє статус HTTP-відповіді. Якщо запит успішний (статус 200–299), нічого не відбувається. Якщо сталася помилка (наприклад, 404 Not Found або 500 Server Error), цей метод автоматично викликає виключення (exception) HTTPError. Це зупиняє подальше виконання коду, якщо дані не були отримані успішно, запобігаючи використанню "битих" даних.

3. Оскільки дані з API приходять як рядок у форматі JSON, метод .json() перетворює цей рядок на зрозумілу для Python структуру даних (список або словник). Тепер course_data — це готовий до роботи об'єкт Python.

4. Метод .extend() бере всі елементи з course_data (який є списком) і додає кожен з них до списку documents як окремий елемент.

Подивимось, як виглядають окремі елементи нашого списку.

In [3]:
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

### Індексація

In [4]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [5]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

В полі boost_dict ми вказуємо важливість тих, чи інших полів. За замовчуванням значення рівяється 1. Це якщо ми його не вказуємо. Запис boost_dict={"question": 2.0, "section": 0.5} означає, що поле question для пошуку найважливіше. Якщо слова з питання будуть знайдені в ньому, ми надамо цим результатам в 4 рази більше значення, ніж якщо вони будуть знайдені в полі section.

Створимо функцію пошуку.

In [6]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

## 2. LLM

### Підключення та тестування моделі

Завантажуємо АРІ-ключ з файлу .env у змінну оточення GEMINI_API_KEY.

In [7]:
from dotenv import load_dotenv
load_dotenv()

True

Створюємо клієнт Gemini та функцію для виклику моделі.

In [8]:
from google import genai

client = genai.Client()

In [19]:
def llm(prompt):
    interaction = client.interactions.create(
        model="gemini-3.1-flash-lite",
        input=prompt
    )
    return interaction.output_text

Перевіримо, як все працює. Зробимо запит до моделі і отримаємо відповідь.

In [20]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

To give you the most accurate answer, **I need a little more information.**

Since I am an AI, I don’t know which specific course you are referring to. Could you please clarify a few things?

1.  **What is the name of the course or the institution/platform offering it?** (e.g., Coursera, Udemy, a specific university program, or an online community).
2.  **Where did you see it?** (Did you receive an email, see an ad, or find it on a website?)

### In the meantime, here is how you can check:

*   **Check the Website:** Go to the official course landing page. If it is still accepting students, there will usually be an "Enroll Now," "Join Course," or "Register" button. 
*   **Look for "Self-Paced" vs. "Cohort-Based":** 
    *   **Self-Paced:** You can almost always join these at any time and work through the materials at your own speed.
    *   **Cohort-Based:** These have specific start and end dates. If you missed the start date, you may have to wait for the next "intake" or session.
*  

### Створення промпту

Промпт складається з двох частин. Перша - це інструкції для нейронки. Друга - безпосередньо запит користувача. Ось інструкції:

In [9]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

А для запиту користувача створимо шаблон.

In [16]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

Створимо функцію для формування контексту.

In [10]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

Об'єднаємо питання з контекстом в користувацький запит.

In [13]:
question = "I just discovered the course. Can I join now?"

In [14]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

Спробуємо, що вийшло.

In [17]:
prompt = build_prompt(question, search_results)

print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

### Запит до моделі

In [18]:
def llm(instructions, user_prompt, model="gemini-3.1-flash-lite"):

    interaction = client.interactions.create(
        model=model,
        system_instruction=instructions,
        input=user_prompt
    )

    return interaction.output_text

## Повний RAG

Об'єднаємо разом БД, конструктор промптів та LLM.

In [19]:
def rag(query, model="gemini-3.1-flash-lite"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

Потестуємо.

In [24]:
answer = rag("I just discovered the course. Can I join now?")
print(answer)

Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [23]:
answer = rag("How do I get a certificate?")
print(answer)

To get a certificate, you must follow these requirements:

*   **Finish the course with a "live" cohort:** You cannot get a certificate if you follow the course in self-paced mode.
*   **Pass the Capstone project:** You must finish and pass the Capstone project. As part of this process, you are required to peer-review 3 capstone projects.
*   **Official Name:** Ensure your official name (as it appears on your identification documents) is entered in the "Edit Course Profile" section. This name will appear on your certificate.

Please note that homework is not mandatory for receiving a certificate.
